In [1]:
from dotenv import load_dotenv    
import os                         

load_dotenv()                    

from langchain_openai import ChatOpenAI                          
from langchain_core.messages import HumanMessage, SystemMessage  
from langgraph.graph import StateGraph, START, END               
from langgraph.graph.message import add_messages                 
from typing import TypedDict, Annotated     

In [2]:
load_dotenv(override=True) 

True

In [3]:
class State(TypedDict):
    messages: Annotated[list, add_messages]   

In [4]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("API_TOKEN"),
    base_url="https://openrouter.ai/api/v1"
)

In [5]:
def chatbot(state: State) -> dict:
    """
    The chatbot node. Takes the current messages,
    sends them to the LLM, and returns the response.
    """
    
    system = SystemMessage(content="You are a helpful and friendly assistant.")
    
    
    response = llm.invoke([system] + state["messages"])
    
    
    return {"messages": [response]}

In [6]:
graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)
graph = graph_builder.compile()


In [7]:
result = graph.invoke({
    "messages": [HumanMessage(content="Who is the US President")]
})

In [9]:
for msg in result["messages"]:
    if hasattr(msg, 'content'):
        role = "Request" if isinstance(msg, HumanMessage) else "AI Response"
        print(f"{role}: {msg.content}\n")

Request: Who is the US President

AI Response: As of my last knowledge update in October 2023, Joe Biden is the President of the United States. He took office on January 20, 2021. If you need the most current information or updates, please verify with a reliable news source.

